# Notebook 02: Exploratory Data Analysis## Spectral Visualization & Label VerificationAnalyze the 947 real UV-Vis spectra.

In [ ]:
import numpy as np, pandas as pdimport matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltimport seaborn as snsfrom pathlib import Pathimport warnings; warnings.filterwarnings('ignore')BASE = Path('/run/media/sham/AI_/ai-stack/projects/biopharma-contamination-detection')OUT = BASE / 'data' / 'processed'FIG = BASE / 'figures'FIG.mkdir(exist_ok=True)plt.rcParams['figure.dpi'] = 150plt.rcParams['savefig.dpi'] = 300sns.set_style('whitegrid')df = pd.read_parquet(OUT / 'real_dataset.parquet')print(f'Loaded {len(df)} spectra')

## Unpack spectra

In [ ]:
import json, numpy as npdef unpack_json(row):    wl = np.array(json.loads(row['wavelengths_json']))    ab = np.array(json.loads(row['absorbance_json']))    return wl, abspectra = []for _, row in df.iterrows():    wl, ab = unpack_json(row)    spectra.append({'wavelengths': wl, 'absorbance': ab, 'label': row['label'],                    'organism': row['organism'], 'cfu': row['cfu'], 'donor_id': row['donor_id'],                    'category': row['category']})df_spec = pd.DataFrame(spectra)print(f'Unpacked {len(df_spec)} spectra')

## 1. Mean spectra by organism

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))for org in df_spec['organism'].unique():    mask = df_spec['organism'] == org    if mask.sum() == 0: continue    abs_arr = np.vstack(df_spec.loc[mask, 'absorbance'].values)    mean_abs = abs_arr.mean(axis=0)    wl = df_spec.loc[mask, 'wavelengths'].iloc[0]    ax.plot(wl, mean_abs, label=f'{org} (n={mask.sum()})', alpha=0.8, linewidth=1.5)ax.set_xlabel('Wavelength (nm)', fontsize=12)ax.set_ylabel('Mean Absorbance', fontsize=12)ax.set_title('Mean UV-Vis Spectra by Organism (947 Real Spectra)', fontsize=14)ax.legend(fontsize=9, ncol=2, loc='best')ax.axvline(260, color='red', alpha=0.3, linestyle='--', label='DNA (260nm)')ax.axvline(280, color='blue', alpha=0.3, linestyle='--', label='Protein (280nm)')ax.axvline(600, color='green', alpha=0.3, linestyle='--', label='Biomass OD600')plt.tight_layout()fig.savefig(FIG / 'spectra_by_organism.png', dpi=300)print('Saved: spectra_by_organism.png')plt.close()

## 2. Sterile vs Contaminated

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))for lbl, cat, ax in [(0, 'Sterile', axes[0]), (1, 'Contaminated', axes[1])]:    mask = df_spec['label'] == lbl    if mask.sum() == 0: continue    abs_arr = np.vstack(df_spec.loc[mask, 'absorbance'].values)    mean = abs_arr.mean(axis=0)    std = abs_arr.std(axis=0)    wl = df_spec.loc[mask, 'wavelengths'].iloc[0]    ax.fill_between(wl, mean-std, mean+std, alpha=0.2)    ax.plot(wl, mean, linewidth=2)    ax.set_title(f'{cat} (n={mask.sum()})', fontsize=13)    ax.set_xlabel('Wavelength (nm)'); ax.set_ylabel('Absorbance')plt.suptitle('Sterile vs Contaminated: Mean ± 1σ', fontsize=15)plt.tight_layout()fig.savefig(FIG / 'sterile_vs_contaminated.png', dpi=300)print('Saved: sterile_vs_contaminated.png')plt.close()

## 3. CFU-level absorbance

In [ ]:
key_wl = {260: 'DNA', 280: 'Protein', 600: 'Biomass'}fig, axes = plt.subplots(1, 3, figsize=(18, 5))for (wl_nm, label), ax in zip(key_wl.items(), axes):    vals = []    for _, row in df_spec[df_spec['cfu'] > 0].iterrows():        idx = np.argmin(np.abs(row['wavelengths'] - wl_nm))        vals.append({'cfu': row['cfu'], 'abs': row['absorbance'][idx], 'organism': row['organism']})    if not vals: continue    df_k = pd.DataFrame(vals)    sns.boxplot(data=df_k, x='cfu', y='abs', ax=ax)    ax.set_title(f'{label} @ {wl_nm}nm', fontsize=12)    ax.set_xlabel('CFU'); ax.set_ylabel('Absorbance')plt.suptitle('Absorbance at Key Wavelengths by CFU Level', fontsize=14)plt.tight_layout()fig.savefig(FIG / 'absorbance_by_cfu.png', dpi=300)print('Saved: absorbance_by_cfu.png')plt.close()

## 4. Data Quality Summary

In [ ]:
print('=== DATA QUALITY ===')print(f'Total spectra: {len(df)}')print(f'Sterile: {(df["label"]==0).sum()}, Contaminated: {(df["label"]==1).sum()}, Timepoint: {(df["category"]=="timepoint").sum()}')print(f'\nOrganisms: {df["organism"].nunique()}')for org in df['organism'].value_counts().index:    n = (df['organism']==org).sum()    print(f'  {org}: {n}')print(f'\nCFU levels: {sorted(df[df["cfu"]>0]["cfu"].unique())}')print(f'Donors: {df["donor_id"].nunique()} ({", ".join(df["donor_id"].value_counts().head(10).index.tolist())})')print(f'\nWavelength range: {df["wl_min"].min():.1f} - {df["wl_max"].max():.1f} nm')print(f'Mean wavelengths per spectrum: {df["n_wl"].mean():.0f}')print('\n✅ Exploratory analysis complete!')